### Import libraries
Spark functions for column operations; DeltaTable for updating Delta tables later.

In [0]:
%python
from pyspark.sql import functions as F
from delta.tables import DeltaTable

### Load utilities
Brings in shared schema names, S3 path and helper functions.

In [0]:

%run ../../01_setup/utilities

### Utilities
Shared settings (catalog, schemas, S3 path) and a helper for full table names. Loaded by every pipeline notebook with `%run`.

In [0]:
%python
print(bronze_schema, silver_schema, gold_schema)

bronze silver gold


### Notebook parameters
Widgets make the notebook reusable: catalog and data source can be changed without editing code.

In [0]:
%python
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "customers", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f"{s3_base_path}/{data_source}/*.csv"
print(base_path)

s3://sportsbar-data-lake-vkt/customers/*.csv


## Bronze

### Read raw CSVs from S3
Loads the files as-is, adding load time, source file name and file size for traceability.

In [0]:
df = (
    spark.read.format("csv")
    .option("header", True)
    .option("inferSchema", True)
    .load(base_path)
    .withColumn("read_timestamp", F.current_timestamp())
    .select("*", "_metadata.file_name", "_metadata.file_size")
)

df.printSchema()
display(df.limit(10))

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = false)
 |-- file_name: string (nullable = false)
 |-- file_size: long (nullable = false)



customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-11-04T16:47:49.699Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-11-04T16:47:49.699Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-11-04T16:47:49.699Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-11-04T16:47:49.699Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-11-04T16:47:49.699Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-11-04T16:47:49.699Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-11-04T16:47:49.699Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-11-04T16:47:49.699Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-11-04T16:47:49.699Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-11-04T16:47:49.699Z,customers.csv,1404


### Save to bronze
Stores the raw data as a Delta table with Change Data Feed on (keeps a record of every row change).

In [0]:
(
    df.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(bronze_schema, data_source))
)

## Silver

### Load bronze data
Reads the raw customers table from bronze as the starting point for cleaning.

In [0]:
df_bronze = spark.table(table_name(bronze_schema, data_source))
df_bronze.printSchema()
display(df_bronze.limit(10))

root
 |-- customer_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



customer_id,customer_name,city,read_timestamp,file_name,file_size
789201,FitFuel Market,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404
789202,FitFuel Market,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404
789203,FitFuel Market,New Delhi,2026-11-04T16:47:56.426Z,customers.csv,1404
789301,Athlete's Choice Store,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404
789303,Athlete's Choice Store,New Delhi,2026-11-04T16:47:56.426Z,customers.csv,1404
789101,Endurance Foods,Bengalore,2026-11-04T16:47:56.426Z,customers.csv,1404
789102,Endurance Foods,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404
789103,Endurance Foods,New Delhi,2026-11-04T16:47:56.426Z,customers.csv,1404
789121,HydroBoost Nutrition,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404
789122,HydroBoost Nutrition,New Delhi,2026-11-04T16:47:56.426Z,customers.csv,1404


### 1. Remove duplicate customers
Shows customer IDs that appear more than once, then keeps one row per ID.

In [0]:
df_duplicates = df_bronze.groupBy("customer_id").count().filter(F.col("count") > 1)
display(df_duplicates)

print("Rows before:", df_bronze.count())
df_silver = df_bronze.dropDuplicates(["customer_id"])
print("Rows after:", df_silver.count())

customer_id,count
789321,2
789503,2
789522,2
789603,2


Rows before: 39
Rows after: 35


### 2. Trim extra spaces in customer names
Removes spaces at the start and end of names (e.g. " Sprintx Nutrition " → "Sprintx Nutrition").

In [0]:
# Names with extra spaces at the start or end
display(df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name"))))

df_silver = df_silver.withColumn("customer_name", F.trim(F.col("customer_name")))

customer_id,customer_name,city,read_timestamp,file_name,file_size
789121,HydroBoost Nutrition,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404
789401,SprintX nutrition,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404
789420,ZenAthlete foods,null,2026-11-04T16:47:56.426Z,customers.csv,1404
789421,ZenAthlete Foods,Hyderbad,2026-11-04T16:47:56.426Z,customers.csv,1404
789521,PrimeFuel Nutrition,null,2026-11-04T16:47:56.426Z,customers.csv,1404
789702,StaminaX Store,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404


### 3. Fix city typos
Maps misspellings to the correct city; anything outside the three known cities becomes empty (null).

In [0]:
df_silver.select("city").distinct().show()

city_mapping = {
    "Bengaluruu": "Bengaluru",
    "Bengalore": "Bengaluru",
    "Hyderabadd": "Hyderabad",
    "Hyderbad": "Hyderabad",
    "NewDelhi": "New Delhi",
    "NewDheli": "New Delhi",
    "NewDelhee": "New Delhi",
}

allowed_cities = ["Bengaluru", "Hyderabad", "New Delhi"]

df_silver = (
    df_silver
    .replace(city_mapping, subset=["city"])
    .withColumn(
        "city",
        F.when(F.col("city").isin(allowed_cities), F.col("city")).otherwise(None)
    )
)

df_silver.select("city").distinct().show()

+----------+
|      city|
+----------+
| Bengaluru|
| Hyderabad|
| New Delhi|
| Bengalore|
|Hyderabadd|
|      NULL|
|  Hyderbad|
| NewDelhee|
|  NewDelhi|
|Bengaluruu|
|  NewDheli|
+----------+

+---------+
|     city|
+---------+
|Bengaluru|
|Hyderabad|
|New Delhi|
|     NULL|
+---------+



### 4. Fix capitalisation in customer names
Converts names to Title Case (e.g. "sprintx nutrition" → "Sprintx Nutrition").

In [0]:
df_silver.select("customer_name").distinct().show()

df_silver = df_silver.withColumn("customer_name", F.initcap("customer_name"))

df_silver.select("customer_name").distinct().show()

+--------------------+
|       customer_name|
+--------------------+
|      FitFuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|HydroBoost Nutrition|
|MacroBite Superfoods|
|MacroBite superfoods|
|      PowerSnack Hub|
|      PowerSnack hub|
|   SprintX nutrition|
|   SprintX Nutrition|
|    ZenAthlete foods|
|    ZenAthlete Foods|
|Peak performance ...|
|Peak Performance ...|
| PrimeFuel Nutrition|
|       Recovery Lane|
|      StaminaX Store|
|EliteAthlete Nutr...|
|      GamePlan Foods|
|   Champion's choice|
+--------------------+
only showing top 20 rows
+--------------------+
|       customer_name|
+--------------------+
|      Fitfuel Market|
|Athlete's Choice ...|
|     Endurance Foods|
|Hydroboost Nutrition|
|Macrobite Superfoods|
|      Powersnack Hub|
|   Sprintx Nutrition|
|    Zenathlete Foods|
|Peak Performance ...|
| Primefuel Nutrition|
|       Recovery Lane|
|      Staminax Store|
|Eliteathlete Nutr...|
|      Gameplan Foods|
|   Champion's Choice|
+--------

### 5. Fill missing cities
Cities for four customers were confirmed by the business team and filled in by customer ID.

In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

customer_city_fix = {
    789403: "New Delhi",   # Sprintx Nutrition
    789420: "Bengaluru",   # Zenathlete Foods
    789521: "Hyderabad",   # Primefuel Nutrition
    789603: "Hyderabad",   # Recovery Lane
}

df_fix = spark.createDataFrame(
    list(customer_city_fix.items()),
    ["customer_id", "fixed_city"]
)

df_silver = (
    df_silver
    .join(df_fix, "customer_id", "left")
    .withColumn("city", F.coalesce("city", "fixed_city"))
    .drop("fixed_city")
)

# Check: these four should now have a city
fixed_names = ["Sprintx Nutrition", "Zenathlete Foods", "Primefuel Nutrition", "Recovery Lane"]
df_silver.filter(F.col("customer_name").isin(fixed_names)).show(truncate=False)

+-----------+-------------------+----+--------------------------+-------------+---------+
|customer_id|customer_name      |city|read_timestamp            |file_name    |file_size|
+-----------+-------------------+----+--------------------------+-------------+---------+
|789403     |Sprintx Nutrition  |NULL|2026-11-04 16:47:56.426954|customers.csv|1404     |
|789420     |Zenathlete Foods   |NULL|2026-11-04 16:47:56.426954|customers.csv|1404     |
|789521     |Primefuel Nutrition|NULL|2026-11-04 16:47:56.426954|customers.csv|1404     |
|789603     |Recovery Lane      |NULL|2026-11-04 16:47:56.426954|customers.csv|1404     |
+-----------+-------------------+----+--------------------------+-------------+---------+

+-----------+-------------------+---------+--------------------------+-------------+---------+
|customer_id|customer_name      |city     |read_timestamp            |file_name    |file_size|
+-----------+-------------------+---------+--------------------------+-------------+-----

### 6. Convert customer_id to text
Matches the parent company, where customer codes are stored as text.

In [0]:
df_silver = df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
df_silver.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- read_timestamp: timestamp (nullable = true)
 |-- file_name: string (nullable = true)
 |-- file_size: long (nullable = true)



### 7. Align with parent company columns
Builds `customer` as "Name-City" and adds market, platform and channel, which SportsBar's data doesn't have.

In [0]:
df_silver = (
    df_silver
    .withColumn(
        "customer",
        F.concat_ws("-", "customer_name", F.coalesce(F.col("city"), F.lit("Unknown")))
    )
    .withColumn("market", F.lit("India"))
    .withColumn("platform", F.lit("Sports Bar"))
    .withColumn("channel", F.lit("Acquisition"))
)

display(df_silver.limit(5))

customer_id,customer_name,city,read_timestamp,file_name,file_size,customer,market,platform,channel
789622,Eliteathlete Nutrition,New Delhi,2026-11-04T16:47:56.426Z,customers.csv,1404,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub,Hyderabad,2026-11-04T16:47:56.426Z,customers.csv,1404,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market,Bengaluru,2026-11-04T16:47:56.426Z,customers.csv,1404,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


### Save to silver
`overwriteSchema` lets the table's columns change on re-runs (e.g. customer_id from number to text).

In [0]:
(
    df_silver.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .option("overwriteSchema", "true")
    .mode("overwrite")
    .saveAsTable(table_name(silver_schema, data_source))
)

## Gold

### Keep business columns and save to gold
Drops technical columns (load time, file details) and saves as `sb_dim_customers`.

In [0]:
df_gold = spark.table(table_name(silver_schema, data_source)).select(
    "customer_id", "customer_name", "city", "customer", "market", "platform", "channel"
)

(
    df_gold.write
    .format("delta")
    .option("delta.enableChangeDataFeed", "true")
    .mode("overwrite")
    .saveAsTable(table_name(gold_schema, f"sb_dim_{data_source}"))
)

## Merge into parent

### Prepare SportsBar customers in the parent's format
Renames `customer_id` to `customer_code` so the columns match Atlikon's `dim_customers`.

In [0]:
df_child_customers = spark.table(table_name(gold_schema, "sb_dim_customers")).select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "market",
    "platform",
    "channel",
)

display(df_child_customers.limit(5))

customer_code,customer,market,platform,channel
789622,Eliteathlete Nutrition-New Delhi,India,Sports Bar,Acquisition
789321,Powersnack Hub-Hyderabad,India,Sports Bar,Acquisition
789601,Recovery Lane-Bengaluru,India,Sports Bar,Acquisition
789720,Gameplan Foods-Bengaluru,India,Sports Bar,Acquisition
789201,Fitfuel Market-Bengaluru,India,Sports Bar,Acquisition


### Check for ID clashes
If a SportsBar ID matches an existing Atlikon code, the merge would overwrite that Atlikon customer. Should be 0.

In [0]:
parent_codes = spark.table(table_name(gold_schema, "dim_customers")).select("customer_code")
child_codes = df_child_customers.select("customer_code")

clashes = parent_codes.intersect(child_codes)
print("Clashing IDs:", clashes.count())
display(clashes)

Clashing IDs: 35


customer_code
789402
789522
789601
789202
789521
789103
789220
789321
789101
789421


### Merge into Atlikon's dim_customers
Updates customers that already exist and inserts new ones. Safe to re-run.

In [0]:
target = DeltaTable.forName(spark, table_name(gold_schema, "dim_customers"))

(
    target.alias("target")
    .merge(
        source=df_child_customers.alias("source"),
        condition="target.customer_code = source.customer_code",
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Verify the merge
Atlikon's original customers plus SportsBar's customers should now be in one table.

In [0]:
dim_customers = spark.table(table_name(gold_schema, "dim_customers"))
print("Total customers:", dim_customers.count())
display(dim_customers.groupBy("platform").count())

Total customers: 53


platform,count
Sports Bar,35
E-Commerce,6
Brick & Mortar,12


### Add an Unknown Customer row
Labels orders with invalid customer IDs (999999) clearly in the dashboard instead of showing a blank customer.

In [0]:
unknown_customer = spark.createDataFrame(
    [("999999", "Unknown Customer", "India", "Sports Bar", "Acquisition")],
    ["customer_code", "customer", "market", "platform", "channel"],
)

(
    DeltaTable.forName(spark, table_name(gold_schema, "dim_customers")).alias("target")
    .merge(unknown_customer.alias("source"), "target.customer_code = source.customer_code")
    .whenNotMatchedInsertAll()
    .execute()
)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
display(spark.table(table_name(gold_schema, "dim_customers")).filter(F.col("customer_code") == "999999"))

customer_code,customer,market,platform,channel
999999,Unknown Customer,India,Sports Bar,Acquisition
